# Resume Analyzer — Full Backend (LLM Upgraded)
**Original rule-based engine fully preserved. LLM layer added on top.**

```
pip install pandas beautifulsoup4 lxml kagglehub
pip install sentence-transformers        # Tier 1 — semantic (offline)
pip install requests                     # Tier 2 — Ollama (local LLM)
pip install huggingface_hub              # Tier 3 — HuggingFace (cloud)
pip install fastapi uvicorn python-multipart   # API server
```


## Cell 1 — Imports & Logging

In [ ]:
import os
import re
import json
import logging
import requests
import pandas as pd
from collections import Counter
from bs4 import BeautifulSoup

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
log = logging.getLogger(__name__)


## Cell 2 — Load Dataset

In [ ]:
import kagglehub

path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")
csv_path = os.path.join(path, "Resume", "Resume.csv")
resume_df = pd.read_csv(csv_path)

log.info(f"Dataset loaded — {resume_df.shape[0]} resumes")
print("Columns   :", resume_df.columns.tolist())
print("Shape     :", resume_df.shape)
print("Categories:", resume_df["Category"].unique().tolist())
resume_df.head(3)


## Cell 3 — Clean Resume Text

In [ ]:
def clean_text(raw: str) -> str:
    """Strip HTML tags and collapse whitespace to plain text."""
    if not isinstance(raw, str):
        return ""
    soup = BeautifulSoup(raw, "lxml")
    text = soup.get_text(separator=" ")
    return re.sub(r"\s+", " ", text).strip()


resume_df["clean_resume"] = resume_df["Resume_str"].apply(clean_text)

print("Sample (first 300 chars):")
print(resume_df["clean_resume"].iloc[0][:300])


## Cell 4 — Build Skill Database from Dataset

In [ ]:
STOPWORDS = {
    "the", "and", "for", "with", "this", "that", "from", "have", "has",
    "been", "will", "was", "are", "were", "not", "but", "all", "any",
    "can", "work", "also", "use", "used", "using", "more", "other",
    "include", "including", "experience", "skills", "ability", "strong",
    "knowledge", "team", "years", "year", "month", "months", "good",
    "well", "within", "across", "our", "your", "their", "such", "both",
    "each", "into", "over", "than", "then", "they", "what", "when",
    "where", "which", "while", "who", "its", "out", "new", "may",
    "must", "per", "etc", "via", "key", "able", "many", "high",
    "level", "related", "basis", "help", "make", "need", "role", "time"
}


def mine_frequent_terms(series: pd.Series, top_n: int = 300) -> list:
    """Return the top-N most frequent meaningful words across all resumes."""
    counter = Counter()
    for text in series.dropna():
        tokens = re.findall(r"[a-zA-Z][a-zA-Z+#.]{2,}", text.lower())
        filtered = [t for t in tokens if t not in STOPWORDS]
        counter.update(filtered)
    return [word for word, _ in counter.most_common(top_n)]


mined_terms = mine_frequent_terms(resume_df["clean_resume"])
print("Top 50 mined terms:", mined_terms[:50])


In [ ]:
# Curated skill list — derived from mined_terms + domain expertise
SKILLS_BY_DOMAIN = {
    "programming": [
        "python", "java", "javascript", "typescript", "c++", "c#", "r",
        "scala", "golang", "ruby", "swift", "kotlin", "php", "bash", "matlab"
    ],
    "data_and_ml": [
        "machine learning", "deep learning", "nlp", "computer vision",
        "data analysis", "data science", "data engineering", "statistics",
        "pandas", "numpy", "scikit-learn", "tensorflow", "pytorch",
        "keras", "xgboost", "opencv", "feature engineering",
        "model deployment", "mlops"
    ],
    "databases": [
        "sql", "mysql", "postgresql", "mongodb", "redis",
        "oracle", "elasticsearch", "sqlite", "nosql", "firebase"
    ],
    "cloud_and_devops": [
        "aws", "azure", "gcp", "docker", "kubernetes", "terraform",
        "jenkins", "ci/cd", "git", "linux", "ansible", "cloud computing"
    ],
    "web_and_api": [
        "react", "angular", "vue", "node.js", "django", "flask",
        "fastapi", "rest api", "graphql", "html", "css", "spring boot"
    ],
    "hr_and_business": [
        "recruitment", "talent acquisition", "onboarding", "payroll",
        "performance management", "employee relations", "hris", "workday",
        "sap hr", "successfactors", "compensation", "benefits administration",
        "labour law", "compliance", "training and development",
        "organizational development", "change management", "workforce planning"
    ],
    "soft_and_tools": [
        "communication", "leadership", "project management", "agile",
        "scrum", "jira", "excel", "powerpoint", "tableau", "power bi",
        "stakeholder management", "problem solving", "critical thinking"
    ],
    "finance_and_accounting": [
        "financial analysis", "accounting", "auditing", "budgeting",
        "forecasting", "tally", "sap fi", "quickbooks", "taxation",
        "risk management"
    ]
}

COMMON_SKILLS = [
    skill
    for domain_skills in SKILLS_BY_DOMAIN.values()
    for skill in domain_skills
]

print(f"Total skills in database: {len(COMMON_SKILLS)}")


## Cell 5 — Rule-Based Skill Extraction (original)

In [ ]:
SYNONYMS = {
    "ml":                          "machine learning",
    "dl":                          "deep learning",
    "ai":                          "machine learning",
    "natural language processing": "nlp",
    "nodejs":                      "node.js",
    "node js":                     "node.js",
    "reactjs":                     "react",
    "react js":                    "react",
    "vuejs":                       "vue",
    "postgres":                    "postgresql",
    "scikit learn":                "scikit-learn",
    "sklearn":                     "scikit-learn",
    "power-bi":                    "power bi",
    "ms excel":                    "excel",
    "microsoft excel":             "excel",
    "ms powerpoint":               "powerpoint",
    "talent management":           "talent acquisition",
    "hiring":                      "recruitment",
    "sourcing":                    "talent acquisition",
}


def normalise(text: str) -> str:
    text = text.lower().strip()
    return SYNONYMS.get(text, text)


def extract_skills(text: str) -> list:
    """
    Rule-based: match text against COMMON_SKILLS using regex.
    Multi-word skills checked first to prevent partial matches.
    Used as fallback when LLM is unavailable.
    """
    normalised = normalise(text)
    sorted_skills = sorted(COMMON_SKILLS, key=lambda s: len(s.split()), reverse=True)

    found = []
    for skill in sorted_skills:
        pattern = r"\b" + re.escape(skill) + r"\b"
        if re.search(pattern, normalised, re.IGNORECASE):
            found.append(skill)

    return list(dict.fromkeys(found))


# Smoke test
print(extract_skills("5 years in recruitment, talent acquisition, payroll and Excel. Also Python and SQL."))


## Cell 6 — Skill Gap Analysis (original)

In [ ]:
def find_missing(user_skills: list, required_skills: list) -> list:
    """Return skills required by the JD that the candidate doesn't have."""
    user_set = {s.lower() for s in user_skills}
    return [skill for skill in required_skills if skill.lower() not in user_set]


## Cell 7 — Skill Dependency Graph (original)

In [ ]:
# skill → list of prerequisites (what to learn first)
SKILL_GRAPH = {
    # ML / Data Science
    "machine learning":         ["python", "statistics", "numpy", "pandas"],
    "deep learning":            ["machine learning", "tensorflow"],
    "nlp":                      ["machine learning", "python"],
    "computer vision":          ["deep learning", "opencv"],
    "mlops":                    ["machine learning", "docker", "ci/cd"],
    "feature engineering":      ["pandas", "statistics"],
    "model deployment":         ["docker", "flask"],
    "data science":             ["python", "statistics", "sql"],
    "data engineering":         ["python", "sql", "cloud computing"],
    # Backend / Cloud
    "docker":                   ["linux"],
    "kubernetes":               ["docker"],
    "terraform":                ["cloud computing"],
    "ci/cd":                    ["git"],
    "aws":                      ["linux", "cloud computing"],
    "azure":                    ["cloud computing"],
    "gcp":                      ["cloud computing"],
    "fastapi":                  ["python"],
    "django":                   ["python"],
    "flask":                    ["python"],
    "spring boot":              ["java"],
    "rest api":                 ["python"],
    # Frontend
    "react":                    ["javascript", "html", "css"],
    "angular":                  ["typescript", "html", "css"],
    "vue":                      ["javascript", "html", "css"],
    "typescript":               ["javascript"],
    # HR
    "recruitment":              ["communication"],
    "talent acquisition":       ["recruitment", "communication"],
    "onboarding":               ["recruitment"],
    "performance management":   ["communication", "excel"],
    "workforce planning":       ["excel", "organizational development"],
    "hris":                     ["excel"],
    "workday":                  ["hris"],
    "sap hr":                   ["hris"],
    "training and development": ["communication", "organizational development"],
    "change management":        ["organizational development", "stakeholder management"],
    "payroll":                  ["excel", "compliance"],
    "benefits administration":  ["compliance", "excel"],
    "compensation":             ["excel", "payroll"],
    # Finance
    "financial analysis":       ["excel", "accounting"],
    "forecasting":              ["financial analysis", "statistics"],
    "auditing":                 ["accounting", "compliance"],
    "risk management":          ["financial analysis", "compliance"],
    "sap fi":                   ["accounting"],
    # BI / Analytics
    "tableau":                  ["excel", "sql"],
    "power bi":                 ["excel", "sql"],
    "elasticsearch":            ["sql", "linux"],
    "postgresql":               ["sql"],
    "mongodb":                  ["sql"],
}


## Cell 8 — Adaptive Learning Path (original)

In [ ]:
def _collect_prereqs(skill: str, known: set, path: list, visited: set) -> None:
    """Depth-first post-order: prerequisites come before the skill itself."""
    if skill in visited:
        return
    visited.add(skill)
    for prereq in SKILL_GRAPH.get(skill, []):
        if prereq.lower() not in known:
            _collect_prereqs(prereq, known, path, visited)
    if skill.lower() not in known and skill not in path:
        path.append(skill)


def generate_learning_path(user_skills: list, missing_skills: list) -> list:
    """
    Ordered learning path — prerequisites injected automatically,
    already-known skills skipped.
    """
    known = {s.lower() for s in user_skills}
    learning_path = []
    visited = set()
    for skill in missing_skills:
        _collect_prereqs(skill, known, learning_path, visited)
    return learning_path


## Cell 9 — Rule-Based Reasoning Engine (original)

In [ ]:
def generate_reasoning(user_skills: list, required_skills: list, missing_skills: list) -> list:
    """
    One explanation per required skill.
    Missing skills get a prereq hint if the candidate already knows related skills.
    """
    reasons = []
    user_set = {s.lower() for s in user_skills}

    for skill in required_skills:
        if skill.lower() in user_set:
            reasons.append({
                "skill": skill.title(),
                "status": "matched",
                "reason": "Found in resume and required by the job description."
            })
        else:
            prereqs = SKILL_GRAPH.get(skill.lower(), [])
            known_prereqs = [p for p in prereqs if p.lower() in user_set]
            hint = (
                f" You already know {', '.join(known_prereqs)} — "
                "prerequisite(s) covered, should be quick to learn."
                if known_prereqs else ""
            )
            reasons.append({
                "skill": skill.title(),
                "status": "missing",
                "reason": f"Required in job description but not found in resume.{hint}"
            })

    return reasons


## Cell 10 — Original analyze() Function

In [ ]:
def analyze(resume_text: str, jd_text: str) -> dict:
    """
    Original rule-based pipeline. Fast, no external dependencies, always works.
    """
    clean_resume = clean_text(resume_text)
    clean_jd     = clean_text(jd_text)

    user_skills     = extract_skills(clean_resume)
    required_skills = extract_skills(clean_jd)
    missing_skills  = find_missing(user_skills, required_skills)
    learning_path   = generate_learning_path(user_skills, missing_skills)
    reasoning       = generate_reasoning(user_skills, required_skills, missing_skills)

    return {
        "mode":            "rule_based",
        "user_skills":     [s.title() for s in user_skills],
        "required_skills": [s.title() for s in required_skills],
        "missing_skills":  [s.title() for s in missing_skills],
        "learning_path":   [s.title() for s in learning_path],
        "reasoning":       reasoning,
    }


## Cell 11 — LLM Upgrade Layer
### Tier 1: Sentence Transformers (offline, no API key)


In [ ]:
# Lazy-load the model so the notebook doesn't block at startup
_semantic_model = None

def _load_semantic_model():
    global _semantic_model
    if _semantic_model is None:
        try:
            from sentence_transformers import SentenceTransformer
            log.info("Loading sentence-transformers model (first run ~30s)...")
            _semantic_model = SentenceTransformer("all-MiniLM-L6-v2")
            log.info("Semantic model ready.")
        except ImportError:
            log.warning("sentence-transformers not installed. pip install sentence-transformers")
    return _semantic_model


def extract_skills_semantic(text: str, threshold: float = 0.42) -> list:
    """
    Upgrade over regex: uses cosine similarity between sentence embeddings
    and skill embeddings. Catches synonyms, paraphrases, and context clues
    that regex misses entirely.
    Falls back to rule-based extract_skills() if model unavailable.
    """
    model = _load_semantic_model()
    if model is None:
        log.warning("Falling back to rule-based skill extraction.")
        return extract_skills(text)

    try:
        from sentence_transformers import util

        sentences = [s.strip() for s in re.split(r"[.;\n]", text) if len(s.strip()) > 10]
        if not sentences:
            return extract_skills(text)

        text_embeddings  = model.encode(sentences, convert_to_tensor=True)
        skill_embeddings = model.encode(COMMON_SKILLS, convert_to_tensor=True)

        found = []
        for i, skill in enumerate(COMMON_SKILLS):
            scores = util.cos_sim(skill_embeddings[i], text_embeddings)
            if scores.max().item() >= threshold:
                found.append(skill)

        # Merge with regex — semantic can miss exact keyword hits
        regex_found = extract_skills(text)
        merged = list(dict.fromkeys(found + regex_found))
        return merged

    except Exception as e:
        log.warning(f"Semantic extraction failed ({e}), falling back to regex.")
        return extract_skills(text)


### Tier 2: Ollama — Local LLM (llama3 / mistral, no internet needed)

In [ ]:
# Setup: https://ollama.com → install → run: ollama pull llama3
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3"   # change to "mistral" or "gemma2" if preferred


def _call_ollama(prompt: str, timeout: int = 60) -> str:
    """Raw call to local Ollama server. Returns response text."""
    resp = requests.post(
        OLLAMA_URL,
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
        timeout=timeout
    )
    resp.raise_for_status()
    return resp.json().get("response", "").strip()


def extract_skills_ollama(text: str) -> list:
    """
    Ask local LLM to extract skills. Much smarter than regex —
    understands context, infers implicit skills, handles typos.
    Falls back to semantic → regex on failure.
    """
    prompt = f"""Extract all professional and technical skills from the text below.
Return ONLY a valid JSON array of skill name strings. No explanation, no markdown.
Example output: ["Python", "SQL", "Recruitment", "Excel"]

Text:
{text[:3000]}

JSON array:"""

    try:
        raw = _call_ollama(prompt)
        start, end = raw.find("["), raw.rfind("]") + 1
        if start == -1:
            raise ValueError("No JSON array in response")
        skills = json.loads(raw[start:end])
        return [s.lower().strip() for s in skills if isinstance(s, str)]
    except Exception as e:
        log.warning(f"Ollama skill extraction failed ({e}), falling back to semantic.")
        return extract_skills_semantic(text)


def generate_reasoning_ollama(user_skills: list, required_skills: list, missing_skills: list) -> list:
    """
    Ask local LLM to write personalised, intelligent gap explanations.
    Much richer than rule-based hints.
    Falls back to rule-based reasoning on failure.
    """
    prompt = f"""You are a professional career coach reviewing a candidate's skill gap.

Candidate's skills: {json.dumps(user_skills)}
Required skills for the job: {json.dumps(required_skills)}
Missing skills: {json.dumps(missing_skills)}

For each required skill, write a short one-sentence explanation (max 20 words).
Be specific, encouraging, and practical.

Return ONLY a valid JSON array of objects. No markdown, no explanation.
Each object must have exactly these keys: "skill", "status", "reason"
"status" must be either "matched" or "missing"

JSON array:"""

    try:
        raw = _call_ollama(prompt, timeout=90)
        start, end = raw.find("["), raw.rfind("]") + 1
        if start == -1:
            raise ValueError("No JSON array in response")
        result = json.loads(raw[start:end])
        for item in result:
            assert "skill" in item and "status" in item and "reason" in item
        return result
    except Exception as e:
        log.warning(f"Ollama reasoning failed ({e}), falling back to rule-based.")
        return generate_reasoning(user_skills, required_skills, missing_skills)


def generate_learning_path_ollama(user_skills: list, missing_skills: list) -> list:
    """
    Ask local LLM to build a smarter, personalised learning path.
    Falls back to graph-based path on failure.
    """
    prompt = f"""You are a learning path designer for professionals.

Candidate already knows: {json.dumps(user_skills)}
Skills the candidate needs to learn: {json.dumps(missing_skills)}

Create an ordered learning path from basic to advanced.
- Start with prerequisites the candidate is missing
- Consider what they already know
- Keep it practical and ordered

Return ONLY a valid JSON array of skill name strings in learning order.
No explanation, no markdown.

JSON array:"""

    try:
        raw = _call_ollama(prompt, timeout=60)
        start, end = raw.find("["), raw.rfind("]") + 1
        if start == -1:
            raise ValueError("No JSON array in response")
        path = json.loads(raw[start:end])
        return [s.strip() for s in path if isinstance(s, str)]
    except Exception as e:
        log.warning(f"Ollama learning path failed ({e}), falling back to graph-based.")
        return generate_learning_path(user_skills, missing_skills)


### Tier 3: HuggingFace Inference API (cloud, no GPU needed)

In [ ]:
# Get free token at huggingface.co → Settings → Access Tokens
HF_TOKEN = os.environ.get("HF_TOKEN", "")   # or paste token directly: HF_TOKEN = "hf_xxx"
HF_MODEL  = "HuggingFaceH4/zephyr-7b-beta"   # or "mistralai/Mixtral-8x7B-Instruct-v0.1"


def extract_skills_huggingface(text: str) -> list:
    """
    Use HuggingFace Inference API for skill extraction.
    Requires HF_TOKEN. Falls back to semantic on failure.
    """
    if not HF_TOKEN:
        log.warning("HF_TOKEN not set. Falling back to semantic extraction.")
        return extract_skills_semantic(text)

    try:
        from huggingface_hub import InferenceClient
        client = InferenceClient(token=HF_TOKEN)

        prompt = f"""<|system|>
You are a resume parser. Extract all professional skills as a JSON array. Return only the array, nothing else.
<|user|>
{text[:2000]}
<|assistant|>"""

        output = client.text_generation(
            prompt,
            model=HF_MODEL,
            max_new_tokens=300,
            temperature=0.1
        )
        start, end = output.find("["), output.rfind("]") + 1
        if start == -1:
            raise ValueError("No JSON array in response")
        skills = json.loads(output[start:end])
        return [s.lower().strip() for s in skills if isinstance(s, str)]

    except Exception as e:
        log.warning(f"HuggingFace extraction failed ({e}), falling back to semantic.")
        return extract_skills_semantic(text)


## Cell 12 — analyze_with_llm() — Upgraded Main Function

In [ ]:
def analyze_with_llm(resume_text: str, jd_text: str, mode: str = "semantic") -> dict:
    """
    LLM-upgraded pipeline. Same output shape as analyze() but smarter.

    mode options:
        'semantic'     — sentence-transformers offline (recommended default)
        'ollama'       — local LLM via Ollama (best quality, needs Ollama running)
        'huggingface'  — HuggingFace Inference API (cloud, needs HF_TOKEN)

    All modes fall back to rule-based if something fails,
    so this function will always return a valid result.
    """
    clean_resume = clean_text(resume_text)
    clean_jd     = clean_text(jd_text)

    log.info(f"Running analyze_with_llm in mode: {mode}")

    # Skill extraction — upgraded
    if mode == "ollama":
        user_skills     = extract_skills_ollama(clean_resume)
        required_skills = extract_skills_ollama(clean_jd)
    elif mode == "huggingface":
        user_skills     = extract_skills_huggingface(clean_resume)
        required_skills = extract_skills_huggingface(clean_jd)
    else:  # default: semantic
        user_skills     = extract_skills_semantic(clean_resume)
        required_skills = extract_skills_semantic(clean_jd)

    # Gap analysis (same logic, smarter inputs)
    missing_skills = find_missing(user_skills, required_skills)

    # Learning path + reasoning (LLM for Ollama, rule-based for others)
    if mode == "ollama":
        learning_path = generate_learning_path_ollama(user_skills, missing_skills)
        reasoning     = generate_reasoning_ollama(user_skills, required_skills, missing_skills)
    else:
        learning_path = generate_learning_path(user_skills, missing_skills)
        reasoning     = generate_reasoning(user_skills, required_skills, missing_skills)

    return {
        "mode":            mode,
        "user_skills":     [s.title() for s in user_skills],
        "required_skills": [s.title() for s in required_skills],
        "missing_skills":  [s.title() for s in missing_skills],
        "learning_path":   [s.title() for s in learning_path],
        "reasoning":       reasoning,
    }


## Cell 13 — Test Both Pipelines

In [ ]:
hr_rows = resume_df[resume_df["Category"] == "HR"]
sample_resume_raw = (
    hr_rows["Resume_str"].iloc[0]
    if not hr_rows.empty
    else resume_df["Resume_str"].iloc[0]
)

sample_jd = """
We are looking for an HR Business Partner with strong experience in
talent acquisition, onboarding, performance management, and HRIS systems
such as Workday. The ideal candidate should have excellent communication
skills, knowledge of labour law, payroll, and compensation. Experience
with training and development and change management is a plus.
Proficiency in Excel and PowerPoint required.
"""

print("=" * 60)
print("TEST 1 — Rule-based (original engine)")
print("=" * 60)
result1 = analyze(sample_resume_raw, sample_jd)
print(json.dumps(result1, indent=2))


In [ ]:
print("=" * 60)
print("TEST 2 — Semantic (sentence-transformers, offline)")
print("=" * 60)
result2 = analyze_with_llm(sample_resume_raw, sample_jd, mode="semantic")
print(json.dumps(result2, indent=2))


In [ ]:
# Uncomment to test Ollama (needs: ollama pull llama3 running locally)
# print("=" * 60)
# print("TEST 3 — Ollama (local LLM)")
# print("=" * 60)
# result3 = analyze_with_llm(sample_resume_raw, sample_jd, mode="ollama")
# print(json.dumps(result3, indent=2))

# Uncomment to test HuggingFace (needs HF_TOKEN set above)
# print("=" * 60)
# print("TEST 4 — HuggingFace API")
# print("=" * 60)
# result4 = analyze_with_llm(sample_resume_raw, sample_jd, mode="huggingface")
# print(json.dumps(result4, indent=2))


## Cell 14 — FastAPI Server (frontend connects here)

In [ ]:
# pip install fastapi uvicorn python-multipart

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Literal

app = FastAPI(
    title="Resume Analyzer API",
    description="AI-powered resume skill gap analysis. Supports rule-based and LLM modes.",
    version="2.0"
)

# Allow all origins during dev — tighten in production
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


class AnalyzeRequest(BaseModel):
    resume_text: str = Field(..., description="Raw resume text (HTML or plain)")
    jd_text:     str = Field(..., description="Job description text")
    mode: Literal["rule_based", "semantic", "ollama", "huggingface"] = Field(
        default="semantic",
        description="rule_based | semantic | ollama | huggingface"
    )


class AnalyzeResponse(BaseModel):
    mode:            str
    user_skills:     list
    required_skills: list
    missing_skills:  list
    learning_path:   list
    reasoning:       list


@app.post("/analyze", response_model=AnalyzeResponse)
def analyze_endpoint(req: AnalyzeRequest):
    """
    Main endpoint. Frontend sends resume + JD, gets back skill analysis.
    """
    if not req.resume_text.strip():
        raise HTTPException(status_code=400, detail="resume_text is empty")
    if not req.jd_text.strip():
        raise HTTPException(status_code=400, detail="jd_text is empty")
    try:
        if req.mode == "rule_based":
            return analyze(req.resume_text, req.jd_text)
        else:
            return analyze_with_llm(req.resume_text, req.jd_text, mode=req.mode)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/health")
def health():
    """Quick health check — frontend pings this to verify server is up."""
    return {"status": "ok", "version": "2.0"}


@app.get("/modes")
def list_modes():
    """
    Tell the frontend which LLM modes are available.
    Frontend can use this to show/hide mode options dynamically.
    """
    semantic_available = True
    try:
        from sentence_transformers import SentenceTransformer  # noqa
    except ImportError:
        semantic_available = False

    ollama_available = False
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        ollama_available = r.status_code == 200
    except Exception:
        pass

    return {
        "rule_based":  True,
        "semantic":    semantic_available,
        "ollama":      ollama_available,
        "huggingface": bool(HF_TOKEN),
    }


@app.get("/dataset/stats")
def dataset_stats():
    """Return basic stats about the loaded dataset — useful for a dashboard."""
    return {
        "total_resumes": len(resume_df),
        "categories":    resume_df["Category"].value_counts().to_dict(),
        "total_skills":  len(COMMON_SKILLS),
    }


print("FastAPI app defined.")
print("To start server, open a terminal and run:")
print("  uvicorn resume_analyzer_llm:app --reload --port 8000")
print("Interactive docs at: http://localhost:8000/docs")


---
## API Quick Reference (for frontend team)

| Endpoint | Method | Purpose |
|---|---|---|
| `/analyze` | POST | Main — send resume + JD, get analysis |
| `/health` | GET | Check if server is alive |
| `/modes` | GET | Which LLM modes are available |
| `/dataset/stats` | GET | Resume count, categories, skill count |

**POST `/analyze` body:**
```json
{
  "resume_text": "...",
  "jd_text": "...",
  "mode": "semantic"
}
```

**Response (all modes return same shape):**
```json
{
  "mode": "semantic",
  "user_skills": [...],
  "required_skills": [...],
  "missing_skills": [...],
  "learning_path": [...],
  "reasoning": [{"skill": "...", "status": "matched/missing", "reason": "..."}]
}
```
